In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

print("All imported")

All imported


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load and prepare data fresh
df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")

# Drop useless columns
df = df.drop(columns=['datetime', 'date', 'station', 'latitude', 'longitude'])

# Encode text columns
day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,'Friday':4,'Saturday':5,'Sunday':6}
season_map = {'monsoon':0,'summer':1,'post_monsoon':2,'winter':3}
df['day_of_week'] = df['day_of_week'].map(day_map)
df['season']      = df['season'].map(season_map)
df['city']        = LabelEncoder().fit_transform(df['city'])

# Separate X and y
X     = df.drop(columns=['aqi','aqi_category'])
y_reg = df['aqi']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (161331, 18)
X_test:  (40333, 18)


In [5]:
lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

print("First 5 predictions:", lr_preds[:5].round())
print("First 5 actual:     ", y_test.values[:5])

First 5 predictions: [ 98. 430. 185. 328. 128.]
First 5 actual:      [117 400 221 284 120]


In [7]:
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)
print("First 5 predictions:", dt_preds[:5].round())
print("First 5 actual:    ", y_test.values[:5])

First 5 predictions: [117. 400. 221. 284. 120.]
First 5 actual:     [117 400 221 284 120]


In [8]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print("First 5 predictions:", rf_preds[:5].round())
print("First 5 actual:    ", y_test.values[:5])

First 5 predictions: [117. 400. 221. 284. 120.]
First 5 actual:     [117 400 221 284 120]


In [10]:
xgb_model = XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("First 5 predictions:", xgb_preds[:5].round())
print("First 5 actual:     ", y_test.values[:5])

First 5 predictions: [118. 401. 220. 284. 120.]
First 5 actual:      [117 400 221 284 120]


In [11]:
from sklearn.metrics import mean_absolute_error, r2_score
models = {
    'Linear Regression': lr_preds,
    'Decision Tree': dt_preds,
    'Random_forest': rf_preds,
    'XGBoost': xgb_preds
}
print(f"{'Model':<22} {'MAE':>8} {'R2 Score':>10}")
print("-"*42)
for name, preds in models.items():
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name:<22} {mae:>8.2f} {r2:>10.4f}")

Model                       MAE   R2 Score
------------------------------------------
Linear Regression         33.51     0.9411
Decision Tree              0.00     1.0000
Random_forest              0.00     1.0000
XGBoost                    0.51     1.0000


In [13]:
print(f"{'Model':<22} {'Train R²':>10} {'Test R²':>10} {'Overfit?':>10}")
print("-" * 55)

train_scores = {
    'Linear Regression': (lr_model, lr_preds),
    'Decision Tree':     (dt_model, dt_preds),
    'Random Forest':     (rf_model, rf_preds),
    'XGBoost':           (xgb_model, xgb_preds)
}

for name, (model, test_preds) in train_scores.items():
    train_r2 = r2_score(y_train, model.predict(X_train))
    test_r2  = r2_score(y_test, test_preds)
    overfit  = "YES " if train_r2 - test_r2 > 0.02 else "No "
    print(f"{name:<22} {train_r2:>10.4f} {test_r2:>10.4f} {overfit:>10}")

Model                    Train R²    Test R²   Overfit?
-------------------------------------------------------
Linear Regression          0.9416     0.9411        No 
Decision Tree              1.0000     1.0000        No 
Random Forest              1.0000     1.0000        No 
XGBoost                    1.0000     1.0000        No 


In [14]:
weather_features = ['year', 'month', 'day', 'hour', 
                    'day_of_week', 'is_weekend', 'season', 'city',
                    'temperature', 'humidity', 'wind_speed', 'visibility']

X_weather = df[weather_features]

X_tr, X_te, y_tr, y_te = train_test_split(X_weather, y_reg, 
                                           test_size=0.2, 
                                           random_state=42)

print("Features used:", weather_features)
print("X_tr shape:", X_tr.shape)

Features used: ['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season', 'city', 'temperature', 'humidity', 'wind_speed', 'visibility']
X_tr shape: (161331, 12)


In [ ]:
lr2  = LinearRegression()
dt2  = DecisionTreeRegressor(random_state=42)
rf2  = RandomForestRegressor(n_estimators=100, random_state=42)
xgb2 = XGBRegressor(n_estimators=100, random_state=42)

lr2.fit(X_tr, y_tr)
dt2.fit(X_tr, y_tr)
rf2.fit(X_tr, y_tr)
xgb2.fit(X_tr, y_tr)

models2 = {
    'Linear Regression': lr2,
    'Decision Tree':     dt2,
    'Random Forest':     rf2,
    'XGBoost':           xgb2
}

print(f"{'Model':<22} {'Train R²':>10} {'Test R²':>10} {'Test MAE':>10}")
print("-" * 55)

for name, model in models2.items():
    train_r2 = r2_score(y_tr, model.predict(X_tr))
    test_r2  = r2_score(y_te, model.predict(X_te))
    test_mae = mean_absolute_error(y_te, model.predict(X_te))
    print(f"{name:<22} {train_r2:>10.4f} {test_r2:>10.4f} {test_mae:>10.2f}")

Model                    Train R²    Test R²   Test MAE
-------------------------------------------------------
Linear Regression          0.9243     0.9233      37.66
Decision Tree              1.0000     0.9480      26.24
Random Forest              0.9962     0.9724      20.32
XGBoost                    0.9769     0.9740      20.05


In [18]:
import joblib

joblib.dump(xgb2, '../models/best_model_xgboost.pkl')
print("Best model saved!")

Best model saved!
